# WiDS 2026 Wildfire Survival — Structural Hazard Ensemble


In [1]:
import os, math, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

SEED = 20260419
np.random.seed(SEED)
NEAR_THRESHOLD_M = 5000.0
HORIZONS = [12, 24, 48]
PROB_COLS = ['prob_12h','prob_24h','prob_48h','prob_72h']


def find_data_dir():
    candidates = [
        '/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26',
        '/kaggle/input/widsworldwide-globaldathon26',
        '/kaggle/input/WiDSWorldWide_GlobalDathon26',
        '/kaggle/input/widsworldwide-globaldatathon26',
        '/kaggle/input', '/mnt/data', '.'
    ]
    for path in candidates:
        if os.path.isdir(path):
            for root, dirs, files in os.walk(path):
                if {'train.csv','test.csv','sample_submission.csv'}.issubset(set(files)):
                    return root
    raise FileNotFoundError('Could not locate train.csv/test.csv/sample_submission.csv')

DATA_DIR = find_data_dir()
train = pd.read_csv(os.path.join(DATA_DIR,'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR,'test.csv'))
sample = pd.read_csv(os.path.join(DATA_DIR,'sample_submission.csv'))


def engineer(df):
    r = pd.DataFrame(index=df.index)
    dist = df['dist_min_ci_0_5h'].clip(lower=1.0).astype(float)
    area = df['area_first_ha'].clip(lower=0.0).astype(float)
    fire_radius = np.sqrt(area * 10000.0 / np.pi)
    closing = df['closing_speed_m_per_h'].clip(lower=0.0).astype(float)
    radial = df['radial_growth_rate_m_per_h'].clip(lower=0.0).astype(float)
    centroid = df['centroid_speed_m_per_h'].clip(lower=0.0).astype(float)
    eff_speed = closing + radial + 0.10 * centroid
    r['low'] = df['low_temporal_resolution_0_5h'].astype(float)
    r['nper'] = df['num_perimeters_0_5h'].astype(float)
    r['log_nper'] = np.log1p(df['num_perimeters_0_5h'].astype(float))
    r['dt'] = df['dt_first_last_0_5h'].astype(float)
    r['time_density'] = df['num_perimeters_0_5h'] / (1.0 + df['dt_first_last_0_5h'])
    r['dist_km'] = dist / 1000.0
    r['log_dist'] = np.log1p(dist)
    r['depth_km'] = np.maximum(NEAR_THRESHOLD_M - dist, 0.0) / 1000.0
    r['gap_km'] = np.maximum(dist - NEAR_THRESHOLD_M, 0.0) / 1000.0
    r['edge_depth_km'] = np.maximum(NEAR_THRESHOLD_M + fire_radius - dist, 0.0) / 1000.0
    r['radius_km'] = fire_radius / 1000.0
    r['area_log'] = np.log1p(area)
    r['growth_log'] = np.log1p(df['area_growth_rate_ha_per_h'].clip(lower=0.0).astype(float))
    r['radial_log'] = np.log1p(radial)
    r['closing_log'] = np.log1p(closing)
    r['centroid_log'] = np.log1p(centroid)
    r['eff_log'] = np.log1p(eff_speed)
    r['align'] = df['alignment_abs'].astype(float)
    r['align_signed'] = df['alignment_cos'].astype(float)
    r['dist_r2'] = df['dist_fit_r2_0_5h'].astype(float)
    r['dist_std_log'] = np.log1p(df['dist_std_ci_0_5h'].clip(lower=0.0).astype(float))
    r['bearing_sin'] = df['spread_bearing_sin'].astype(float)
    r['bearing_cos'] = df['spread_bearing_cos'].astype(float)
    hour = df['event_start_hour'].astype(float)
    month = df['event_start_month'].astype(float)
    dow = df['event_start_dayofweek'].astype(float)
    r['hour'] = hour
    r['month'] = month
    r['dow'] = dow
    r['hour_sin'] = np.sin(2*np.pi*hour/24.0)
    r['hour_cos'] = np.cos(2*np.pi*hour/24.0)
    r['month_sin'] = np.sin(2*np.pi*(month-1)/12.0)
    r['month_cos'] = np.cos(2*np.pi*(month-1)/12.0)
    r['dow_sin'] = np.sin(2*np.pi*dow/7.0)
    r['dow_cos'] = np.cos(2*np.pi*dow/7.0)
    r['evening'] = hour.between(18, 22).astype(float)
    r['night_morning'] = ((hour <= 6) | (hour == 9)).astype(float)
    r['summer'] = df['event_start_month'].isin([6,7,8]).astype(float)
    r['peak_aug'] = (df['event_start_month'] == 8).astype(float)
    r['low_x_area'] = r['low'] * r['area_log']
    r['low_x_hour'] = r['low'] * r['hour']
    r['low_x_dist'] = r['low'] * r['dist_km']
    r['dt_x_align'] = r['dt'] * r['align']
    r['nper_x_align'] = r['log_nper'] * r['align']
    r['speed_x_align'] = r['eff_log'] * (0.25 + r['align'])
    r = r.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return r


def add_bins(train_df, test_df):
    tr = train_df.copy().reset_index(drop=True)
    te = test_df.copy().reset_index(drop=True)
    tr['hour_group'] = pd.cut(tr['event_start_hour'], bins=[-1,6,12,18,24], labels=False).astype(int)
    te['hour_group'] = pd.cut(te['event_start_hour'], bins=[-1,6,12,18,24], labels=False).astype(int)
    tr['month_group'] = pd.cut(tr['event_start_month'], bins=[0,5,6,7,8,9,12], labels=False).astype(int)
    te['month_group'] = pd.cut(te['event_start_month'], bins=[0,5,6,7,8,9,12], labels=False).astype(int)
    for src_col, new_col, q in [
        ('area_first_ha','area_group',4),
        ('dist_min_ci_0_5h','dist_group',4),
        ('event_start_hour','hour_rank_group',4),
    ]:
        x = np.asarray(tr[src_col], dtype=float)
        try:
            edges = np.unique(np.quantile(x, np.linspace(0,1,q+1)))
            edges[0] = -np.inf; edges[-1] = np.inf
            if len(edges) <= 2:
                tr[new_col] = 0; te[new_col] = 0
            else:
                tr[new_col] = pd.cut(tr[src_col], bins=edges, labels=False, include_lowest=True).astype(int)
                te[new_col] = pd.cut(te[src_col], bins=edges, labels=False, include_lowest=True).astype(int)
        except Exception:
            tr[new_col] = 0; te[new_col] = 0
    for col in ['hour_group','month_group','area_group','dist_group','hour_rank_group']:
        tr[col] = tr[col].fillna(-1).astype(int)
        te[col] = te[col].fillna(-1).astype(int)
    return tr, te


def robust_standardize(X_train, X_test):
    Xtr = np.asarray(X_train, dtype=float)
    Xte = np.asarray(X_test, dtype=float)
    med = np.nanmedian(Xtr, axis=0)
    q75 = np.nanpercentile(Xtr, 75, axis=0)
    q25 = np.nanpercentile(Xtr, 25, axis=0)
    scale = q75 - q25
    fallback = np.nanstd(Xtr, axis=0)
    scale = np.where(scale < 1e-8, fallback, scale)
    scale = np.where(scale < 1e-8, 1.0, scale)
    return (Xtr - med) / scale, (Xte - med) / scale


def kernel_oof_test(X_train, X_test, y, bandwidth=1.5, smooth=8.0, col_weight=None):
    Ztr, Zte = robust_standardize(X_train, X_test)
    if col_weight is not None:
        w = np.asarray(col_weight, dtype=float).reshape(1, -1)
        Ztr = Ztr * w
        Zte = Zte * w
    dtr = ((Ztr[:, None, :] - Ztr[None, :, :]) ** 2).mean(axis=2)
    W = np.exp(-dtr / (2.0 * bandwidth * bandwidth))
    np.fill_diagonal(W, 0.0)
    prior = float(np.mean(y))
    oof = (W @ y + smooth * prior) / (W.sum(axis=1) + smooth)
    if len(Zte):
        dte = ((Zte[:, None, :] - Ztr[None, :, :]) ** 2).mean(axis=2)
        Wt = np.exp(-dte / (2.0 * bandwidth * bandwidth))
        pred = (Wt @ y + smooth * prior) / (Wt.sum(axis=1) + smooth)
    else:
        pred = np.array([], dtype=float)
    return np.clip(oof, 0, 1), np.clip(pred, 0, 1)


def group_oof_test(train_aug, test_aug, y, cols, smooth=8.0):
    prior = float(np.mean(y))
    n = len(train_aug)
    oof = np.zeros(n, dtype=float)
    keys_train = list(map(tuple, train_aug[cols].to_numpy()))
    keys_test = list(map(tuple, test_aug[cols].to_numpy())) if len(test_aug) else []
    y = np.asarray(y, dtype=float)
    for i, key in enumerate(keys_train):
        idx = [j for j, k in enumerate(keys_train) if k == key and j != i]
        if idx:
            oof[i] = (y[idx].sum() + smooth * prior) / (len(idx) + smooth)
        else:
            oof[i] = prior
    pred = np.zeros(len(test_aug), dtype=float)
    for i, key in enumerate(keys_test):
        idx = [j for j, k in enumerate(keys_train) if k == key]
        if idx:
            pred[i] = (y[idx].sum() + smooth * prior) / (len(idx) + smooth)
        else:
            pred[i] = prior
    return np.clip(oof,0,1), np.clip(pred,0,1)


def one_d_score_oof_test(score_train, score_test, y, bandwidth=0.85, smooth=6.0):
    st = np.asarray(score_train, dtype=float)
    sv = np.asarray(score_test, dtype=float)
    mu = np.nanmedian(st)
    sc = np.nanpercentile(st,75) - np.nanpercentile(st,25)
    if not np.isfinite(sc) or sc < 1e-8:
        sc = np.nanstd(st) + 1e-6
    zt = (st - mu) / sc
    zv = (sv - mu) / sc if len(sv) else np.array([])
    D = (zt[:,None] - zt[None,:])**2
    W = np.exp(-D/(2*bandwidth*bandwidth))
    np.fill_diagonal(W,0.0)
    prior = float(np.mean(y))
    oof = (W @ y + smooth*prior)/(W.sum(axis=1)+smooth)
    if len(zv):
        Dv = (zv[:,None] - zt[None,:])**2
        Wv = np.exp(-Dv/(2*bandwidth*bandwidth))
        pred = (Wv @ y + smooth*prior)/(Wv.sum(axis=1)+smooth)
    else:
        pred = np.array([], dtype=float)
    return np.clip(oof,0,1), np.clip(pred,0,1)


def logit_mean_match(p_train, p_test, target):
    p_train = np.clip(np.asarray(p_train, dtype=float), 1e-5, 1-1e-5)
    p_test = np.clip(np.asarray(p_test, dtype=float), 1e-5, 1-1e-5)
    ztr = np.log(p_train/(1-p_train))
    zte = np.log(p_test/(1-p_test)) if len(p_test) else p_test
    lo, hi = -8.0, 8.0
    for _ in range(60):
        mid = (lo+hi)/2
        m = (1/(1+np.exp(-(ztr+mid)))).mean()
        if m < target:
            lo = mid
        else:
            hi = mid
    delta = (lo+hi)/2
    return 1/(1+np.exp(-(ztr+delta))), 1/(1+np.exp(-(zte+delta))) if len(p_test) else p_test


def blend_components(P_oof, P_test, y):
    P_oof = np.clip(np.asarray(P_oof, dtype=float), 1e-5, 1-1e-5)
    P_test = np.clip(np.asarray(P_test, dtype=float), 1e-5, 1-1e-5)
    y = np.asarray(y, dtype=float)
    losses = np.mean((P_oof - y[:,None])**2, axis=0)
    order = np.argsort(losses)
    top = order[:min(8, len(order))]
    best = order[0]
    spread = max(float(np.std(losses[top])), 0.0025)
    w_soft = np.exp(-(losses[top] - losses[best]) / (0.75*spread + 1e-9))
    w_soft = w_soft / w_soft.sum()
    soft_oof = P_oof[:, top] @ w_soft
    soft_test = P_test[:, top] @ w_soft
    inv = 1.0 / (losses[top] + 0.035)
    inv = inv / inv.sum()
    inv_oof = P_oof[:, top] @ inv
    inv_test = P_test[:, top] @ inv
    raw_oof = 0.05 * P_oof[:, best] + 0.475 * soft_oof + 0.475 * inv_oof
    raw_test = 0.05 * P_test[:, best] + 0.475 * soft_test + 0.475 * inv_test
    shifted_oof, shifted_test = logit_mean_match(raw_oof, raw_test, float(y.mean()))
    if np.mean((shifted_oof - y)**2) <= np.mean((raw_oof - y)**2) + 0.0003:
        final_oof, final_test = shifted_oof, shifted_test
    else:
        final_oof, final_test = raw_oof, raw_test
    return np.clip(final_oof,0,1), np.clip(final_test,0,1), losses[best], best, top.tolist()


def make_components(train_near, test_near, y, h):
    Xtr = engineer(train_near).reset_index(drop=True)
    Xte = engineer(test_near).reset_index(drop=True)
    tr_aug, te_aug = add_bins(train_near.reset_index(drop=True), test_near.reset_index(drop=True))
    comps_oof, comps_test, names = [], [], []
    feature_sets = [
        ['low','log_nper','dt','dist_km','depth_km','area_log','align','hour_sin','hour_cos','month_sin','month_cos','evening','night_morning','summer'],
        ['low','log_nper','dt','area_log','growth_log','radial_log','closing_log','centroid_log','align','dist_r2','dist_std_log','hour','month'],
        ['low','time_density','dist_km','depth_km','edge_depth_km','radius_km','area_log','speed_x_align','dt_x_align','nper_x_align','peak_aug','night_morning'],
        ['low','low_x_area','low_x_hour','low_x_dist','area_log','hour','month','dist_km','depth_km'],
    ]
    bws = [0.70, 1.00, 1.45, 2.10, 3.25]
    for fs_i, fs in enumerate(feature_sets):
        fs = [c for c in fs if c in Xtr.columns]
        for bw in bws:
            o, p = kernel_oof_test(Xtr[fs].values, Xte[fs].values, y, bandwidth=bw, smooth=3.0 + 3.5*bw)
            comps_oof.append(o); comps_test.append(p); names.append(f'kernel{fs_i}_bw{bw}')
    group_specs = [
        (['low_temporal_resolution_0_5h'], 3.0),
        (['low_temporal_resolution_0_5h','hour_group'], 8.0),
        (['low_temporal_resolution_0_5h','area_group'], 8.0),
        (['low_temporal_resolution_0_5h','month_group'], 9.0),
        (['low_temporal_resolution_0_5h','dist_group'], 9.0),
        (['low_temporal_resolution_0_5h','hour_group','area_group'], 13.0),
        (['low_temporal_resolution_0_5h','month_group','area_group'], 14.0),
    ]
    for cols, sm in group_specs:
        o, p = group_oof_test(tr_aug, te_aug, y, cols, smooth=sm)
        comps_oof.append(o); comps_test.append(p); names.append('group_'+'_'.join(cols))
    # Manual rank scores: several deliberately different priors for early timing.
    low = train_near['low_temporal_resolution_0_5h'].astype(float).reset_index(drop=True)
    low_t = test_near['low_temporal_resolution_0_5h'].astype(float).reset_index(drop=True) if len(test_near) else pd.Series([], dtype=float)
    def score_block(df):
        return pd.DataFrame({
            's1': (1-df.low_temporal_resolution_0_5h)*1.60 + np.log1p(df.num_perimeters_0_5h)*0.25 + df.alignment_abs*0.40 + np.log1p(df.area_first_ha)/9.0 + df.event_start_hour.isin([0,1,2,3,4,5,6,9,22]).astype(float)*0.25 - df.dist_min_ci_0_5h/13000.0,
            's2': (1-df.low_temporal_resolution_0_5h)*1.25 + df.dt_first_last_0_5h*0.22 + np.log1p(df.area_first_ha)/7.5 + np.log1p(df.radial_growth_rate_m_per_h.clip(lower=0))*0.22 + df.alignment_abs*0.33 - df.dist_min_ci_0_5h/15000.0,
            's3': -df.low_temporal_resolution_0_5h*1.15 + np.log1p(df.num_perimeters_0_5h)*0.45 + df.event_start_month.eq(8).astype(float)*0.28 + df.event_start_hour.le(9).astype(float)*0.22 + np.log1p(df.area_first_ha)/10.0,
            's4': (5000.0-df.dist_min_ci_0_5h)/2300.0 + np.log1p(df.area_first_ha)/8.0 + np.log1p(df.area_growth_rate_ha_per_h.clip(lower=0))*0.18 + np.log1p(df.centroid_speed_m_per_h.clip(lower=0))*0.18 - df.low_temporal_resolution_0_5h*0.80,
        })
    S_tr = score_block(train_near.reset_index(drop=True))
    S_te = score_block(test_near.reset_index(drop=True)) if len(test_near) else pd.DataFrame(columns=S_tr.columns)
    for col in S_tr.columns:
        for bw in [0.55, 0.85, 1.25, 1.80]:
            o, p = one_d_score_oof_test(S_tr[col].values, S_te[col].values if len(S_te) else np.array([]), y, bandwidth=bw, smooth=5.0 + 2.0*bw)
            comps_oof.append(o); comps_test.append(p); names.append(f'score_{col}_bw{bw}')
    P_oof = np.vstack(comps_oof).T
    P_test = np.vstack(comps_test).T if len(test_near) else np.zeros((0, len(comps_oof)))
    return P_oof, P_test, names


def compute_metric(time, event, probs):
    risk = 0.3*probs[:,1] + 0.4*probs[:,2] + 0.3*probs[:,3]
    conc = comp = 0.0
    for i in range(len(time)):
        if event[i] != 1:
            continue
        for j in range(len(time)):
            if i == j or time[i] >= time[j]:
                continue
            comp += 1
            if risk[i] > risk[j]: conc += 1
            elif risk[i] == risk[j]: conc += 0.5
    cidx = conc / comp if comp else 0.5
    briers = []
    for col, h in zip([1,2,3],[24,48,72]):
        valid = ~((event == 0) & (time < h))
        y = ((event == 1) & (time <= h)).astype(float)[valid]
        briers.append(float(np.mean((probs[:,col][valid] - y)**2)))
    wb = 0.3*briers[0] + 0.4*briers[1] + 0.3*briers[2]
    return 0.3*cidx + 0.7*(1-wb), cidx, wb, briers

near_train_mask = train['dist_min_ci_0_5h'].values < NEAR_THRESHOLD_M
near_test_mask = test['dist_min_ci_0_5h'].values < NEAR_THRESHOLD_M
train_near = train.loc[near_train_mask].reset_index(drop=True)
test_near = test.loc[near_test_mask].reset_index(drop=True)

train_probs = np.zeros((len(train), 4), dtype=float)
test_probs = np.zeros((len(test), 4), dtype=float)
train_probs[:,3] = 1.0
test_probs[:,3] = 1.0
model_report = {}
for col_idx, h in enumerate(HORIZONS):
    y = (train_near['time_to_hit_hours'].values <= h).astype(float)
    P_oof, P_test, names = make_components(train_near, test_near, y, h)
    pred_oof, pred_test, best_loss, best_idx, top = blend_components(P_oof, P_test, y)
    train_probs[near_train_mask, col_idx] = pred_oof
    test_probs[near_test_mask, col_idx] = pred_test
    model_report[str(h)] = {
        'base_rate': float(y.mean()),
        'best_component': names[int(best_idx)],
        'best_brier': float(best_loss),
        'top_components': [names[i] for i in top],
    }

# Structural impossible-event gate outside 5km. Train is perfectly separated by this threshold.
# Keep a tiny halo for numerical robustness and for fires just outside the threshold.
dist_test = test['dist_min_ci_0_5h'].values.astype(float)
gap = np.maximum(dist_test - NEAR_THRESHOLD_M, 0.0)
closing = np.maximum(test['closing_speed_m_per_h'].values.astype(float), 0.0)
radial = np.maximum(test['radial_growth_rate_m_per_h'].values.astype(float), 0.0)
align = np.abs(test['alignment_abs'].values.astype(float))
threat = np.clip((closing + radial) * (0.25 + align) / (gap + 2500.0), 0.0, 1.0)
far_floor = np.vstack([
    1e-6 + 0.0007*np.exp(-gap/2000.0) + 0.0009*threat,
    2e-6 + 0.0013*np.exp(-gap/3500.0) + 0.0016*threat,
    3e-6 + 0.0023*np.exp(-gap/5500.0) + 0.0024*threat,
]).T
for j in range(3):
    test_probs[~near_test_mask, j] = far_floor[~near_test_mask, j]
    train_probs[~near_train_mask, j] = 0.0

train_probs = np.maximum.accumulate(np.clip(train_probs, 0.0, 1.0), axis=1)
test_probs = np.maximum.accumulate(np.clip(test_probs, 0.0, 1.0), axis=1)
train_probs[:,3] = 1.0
test_probs[:,3] = 1.0

# Monotone validation only; no extra hand-forcing after the OOF-selected blend.
train_probs = np.maximum.accumulate(np.clip(train_probs, 0.0, 1.0), axis=1)
test_probs = np.maximum.accumulate(np.clip(test_probs, 0.0, 1.0), axis=1)
train_probs[:,3] = 1.0
test_probs[:,3] = 1.0

score, cidx, wb, briers = compute_metric(train['time_to_hit_hours'].values, train['event'].values.astype(int), train_probs)
print('DATA_DIR:', DATA_DIR)
print('train/test:', train.shape, test.shape)
print('structural separation train: near<5km events =', int(train.loc[near_train_mask,'event'].sum()), '/', int(near_train_mask.sum()), '| far events =', int(train.loc[~near_train_mask,'event'].sum()), '/', int((~near_train_mask).sum()))
print('OOF diagnostic hybrid:', round(score,6), 'c-index:', round(cidx,6), 'weighted_brier:', round(wb,6), 'briers24/48/72:', [round(x,6) for x in briers])
print('model report:', json.dumps(model_report, indent=2))

submission = pd.DataFrame({
    'event_id': test['event_id'].values,
    'prob_12h': test_probs[:,0],
    'prob_24h': test_probs[:,1],
    'prob_48h': test_probs[:,2],
    'prob_72h': test_probs[:,3],
})
submission = sample[['event_id']].merge(submission, on='event_id', how='left', validate='one_to_one')
submission[PROB_COLS] = submission[PROB_COLS].astype(float).clip(0.0, 1.0)
vals = submission[PROB_COLS].values
vals = np.maximum.accumulate(vals, axis=1)
vals[:,3] = 1.0
submission[PROB_COLS] = vals
assert list(submission.columns) == ['event_id'] + PROB_COLS
assert len(submission) == len(sample)
assert submission['event_id'].equals(sample['event_id'])
assert submission['event_id'].nunique() == len(submission)
assert np.isfinite(submission[PROB_COLS].values).all()
assert ((submission[PROB_COLS].values >= 0) & (submission[PROB_COLS].values <= 1)).all()
assert np.all(np.diff(submission[PROB_COLS].values, axis=1) >= -1e-12)
out_dir = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
os.makedirs(out_dir, exist_ok=True)
submission.to_csv(os.path.join(out_dir, 'submission.csv'), index=False)
try:
    submission.to_csv('/mnt/data/submission.csv', index=False)
except Exception:
    pass
print('saved:', os.path.join(out_dir, 'submission.csv'))
print(submission.head(10).to_string(index=False))


DATA_DIR: /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
train/test: (221, 37) (95, 35)
structural separation train: near<5km events = 69 / 69 | far events = 0 / 152
OOF diagnostic hybrid: 0.973467 c-index: 0.944187 weighted_brier: 0.013984 briers24/48/72: [0.026428, 0.01514, 0.0]
model report: {
  "12": {
    "base_rate": 0.7101449275362319,
    "best_component": "group_low_temporal_resolution_0_5h_hour_group",
    "best_brier": 0.14570536141139576,
    "top_components": [
      "group_low_temporal_resolution_0_5h_hour_group",
      "group_low_temporal_resolution_0_5h",
      "score_s3_bw0.55",
      "score_s1_bw0.55",
      "score_s2_bw0.55",
      "group_low_temporal_resolution_0_5h_dist_group",
      "group_low_temporal_resolution_0_5h_area_group",
      "group_low_temporal_resolution_0_5h_month_group"
    ]
  },
  "24": {
    "base_rate": 0.9130434782608695,
    "best_component": "group_low_temporal_resolution_0_5h_hour_group_area_group",
    "best_brier": 0.0737035952871